# 07 - Do the explanations agree? Seeds, models, years, universes

Step 7 of `PROGETTO.md` (PROGETTO.md C2), the central result of the thesis: **does the instability come from the data or from the models?**

Every run of the core grid (6 models x DJI, NASDAQ-100, EURO STOXX 50 x test years 2020-2024 x seeds 0, 5, 42, T = 20, L = 5) is explained with Integrated Gradients (notebook 05, `scripts/explain.py`), and its global importance is a vector over features, lags or families. Two runs that differ in **one factor only** are compared (`xaifin.analysis.variance.pair_agreement`):

| Factor | Pair | Measures |
|---|---|---|
| seed | same model, universe, year; another seed | training noise |
| model | same universe, year, seed; another model | architecture |
| year | same model, universe, seed; another year | non-stationarity of the data |
| universe | same model, year, seed; another universe | universe effect |

Agreement: Spearman, Kendall, Jaccard of the top 10 and rank-biased overlap (`analysis/agreement.py`). The two feature groups share no feature, so models of different groups are compared only on the families both have (momentum, volume, candle). A variance decomposition (`decomposition`) splits each family's importance across runs into model, universe, year, their interactions, and seed.

**Diagnosis logic** (guide, Step 7): models agree and IC is low for all → the data carry little signal (*data-driven*); models disagree with similar IC → several equally good explanations (*Rashomon*, model-driven); seeds disagree as much as models → the explanations are not stable and no architectural conclusion can be drawn.

In [1]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.analysis.agreement import agreement, agreement_matrix
from xaifin.analysis.variance import KEYS, decomposition, load_importance, pair_agreement, vectors
from xaifin.config import CORE_UNIVERSES, UNIVERSE_NAMES
from xaifin.data.features import market_names
from xaifin.training.rolling import collect_results

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
MODELS, PRED_LEN = viz.MODEL_ORDER, 5
METHOD = "ig_joint"
FAMILY_VALUE = "net"  # family shares: |sum of attributions| (= what Shapley measures, notebook 05); "importance" for sum |a|

def load(level, target="pred", value="importance", method=METHOD):
    t = load_importance(method, target, level, value)
    return t[(t["pred_len"] == PRED_LEN) & t["universe"].isin(CORE_UNIVERSES)]

family = {t: load("family", t, FAMILY_VALUE) for t in ("pred", "topk_margin")}
family = {t: f[f["name"] != "market"] for t, f in family.items()}  # zero by construction with the day-mean baseline
feature = {t: load("feature", t) for t in ("pred", "topk_margin")}
market = {(u, n) for u in CORE_UNIVERSES for n in market_names(u)}
feature = {t: f[[(u, n) not in market for u, n in zip(f["universe"], f["name"])]] for t, f in feature.items()}  # as the market family
lag = load("lag")
runs = family["pred"].groupby(["model", "universe", "seed"])["test_year"].nunique().unstack(["universe", "seed"])
print(f"{family['pred'].groupby(['model', 'universe', 'seed', 'test_year']).ngroups} explained runs (L = {PRED_LEN}); test years per cell:")
runs.reindex(MODELS)

252 explained runs (L = 5); test years per cell:


universe    dji       nasdaq100       sx5e      
seed         0  5  42        0  5  42   0  5  42
model                                           
MASTER        5  5  5         5  5  5    4  4  4
MATCC         5  5  5         5  5  5    4  4  4
FactorVAE     5  5  5         5  5  5    4  4  4
HIST          5  5  5         5  5  5    4  4  4
DiscoverPLF   5  5  5         5  5  5    4  4  4
FinFormer     5  5  5         5  5  5    4  4  4

## 1. Agreement by factor

Every pair of runs that differ in one factor only. Families (all models; across groups on the shared families only) and features (within a feature group).

In [2]:
pairs = {"family": pair_agreement(family["pred"]), "feature": pair_agreement(feature["pred"])}
summary = pd.concat({level: p.groupby("factor")[["spearman", "kendall", "jaccard", "rbo"]].mean()
                     .assign(pairs=p.groupby("factor").size()) for level, p in pairs.items()})
display(summary)
both = pd.concat([p.assign(level=level) for level, p in pairs.items()])
px.box(both, x="factor", y="spearman", color="level", category_orders={"factor": ["seed", "model", "year", "universe"]},
       title="Agreement of the importance rankings between two runs differing in one factor (Spearman)").show()

spearman  kendall  jaccard   rbo  pairs
        factor                                           
family  model        0.934    0.903    1.000 0.972    630
        seed         0.915    0.864    1.000 0.955    252
        universe     0.895    0.837    1.000 0.946    234
        year         0.920    0.869    1.000 0.954    468
feature model        0.263    0.185    0.094 0.156    252
        seed         0.590    0.467    0.313 0.401    252
        universe     0.598    0.463    0.282 0.374    234
        year         0.658    0.508    0.303 0.403    468

## 2. Which models explain alike?

Per universe, the family vector of each model (mean over seeds and years), compared pairwise. Within the Alpha158 group (MASTER, MATCC, FactorVAE) all seven families; across groups the three shared ones.

In [3]:
for universe in CORE_UNIVERSES:
    f = family["pred"][family["pred"]["universe"] == universe]
    if f.empty:
        continue
    model_vectors = {m: g.groupby("name")["importance"].mean() for m, g in f.groupby("model")}
    model_vectors = {m: model_vectors[m] for m in MODELS if m in model_vectors}
    matrix = agreement_matrix(model_vectors, "spearman")
    viz.overlap_heatmap(matrix, f"{UNIVERSE_NAMES[universe]}: agreement of the family rankings between models (Spearman)").show()

## 3. Variance decomposition

Share of the variance of each family's importance (across runs) explained by the model, the universe, the test year, their interactions, and the seed (spread of the three seeds within each model x universe x year cell). All models on the shared families, and the Alpha158 group on its seven families.

In [4]:
tables = {"all models, shared families": decomposition(family["pred"]),
          "Alpha158 group, its families": decomposition(family["pred"][family["pred"]["group"] == "alpha158"]),
          "Alpha360 group, its families": decomposition(family["pred"][family["pred"]["group"] == "alpha360"])}
for name, table in tables.items():
    print(name)
    display(table.set_index("name")[["mean_importance", "model", "universe", "year", "interactions", "seed", "runs"]])

all models, shared families


,mean_importance,model,universe,year,interactions,seed,runs
name,,,,,,,
candle,0.055,0.785,0.017,0.012,0.095,0.091,252
momentum,0.454,0.980,0.000,0.001,0.009,0.010,252
volume,0.192,0.798,0.007,0.011,0.087,0.098,252
all (weighted),NaN,0.915,0.003,0.004,0.037,0.041,252


Alpha158 group, its families


,mean_importance,model,universe,year,interactions,seed,runs
name,,,,,,,
candle,0.080,0.065,0.003,0.201,0.315,0.415,126
momentum,0.186,0.075,0.028,0.245,0.268,0.383,126
position,0.198,0.055,0.066,0.005,0.307,0.568,126
price_volume,0.112,0.233,0.192,0.032,0.404,0.139,126
trend,0.151,0.167,0.039,0.111,0.089,0.595,126
volatility,0.137,0.125,0.047,0.019,0.347,0.462,126
volume,0.135,0.036,0.038,0.046,0.551,0.328,126
all (weighted),NaN,0.103,0.058,0.092,0.317,0.430,126


Alpha360 group, its families


,mean_importance,model,universe,year,interactions,seed,runs
name,,,,,,,
candle,0.029,0.478,0.130,0.002,0.178,0.213,126
momentum,0.722,0.470,0.013,0.061,0.172,0.284,126
volume,0.248,0.552,0.018,0.062,0.142,0.225,126
all (weighted),NaN,0.490,0.018,0.060,0.165,0.267,126


## 4. Consistently dominant features

Within each feature group: the features in the top 10 of IG importance in at least half of the explained runs.

In [5]:
f = feature["pred"].copy()
f["rank"] = f.groupby(["model", "universe", "seed", "test_year"])["importance"].rank(ascending=False)
top = (f.assign(top10=f["rank"] <= 10).groupby(["group", "name"])
       .agg(share_top10=("top10", "mean"), mean_rank=("rank", "mean"), models=("model", "nunique")).reset_index())
for group, g in top.groupby("group"):
    print(group)
    display(g[g["share_top10"] >= 0.5].sort_values("share_top10", ascending=False).head(15))

alpha158


,group,name,share_top10,mean_rank,models


alpha360


,group,name,share_top10,mean_rank,models
512,alpha360,VWAP59,0.770,8.119,3
516,alpha360,VWAP9,0.563,14.484,3


## 5. Score or portfolio? `pred` against `topk_margin`, and permutation importance

IG of the score and of the top-k margin, run by run; and permutation importance, whose two measures separate what a family is worth for **ranking skill** (drop in RankIC when shuffled) from what it does to the **portfolio** (share of the top-k replaced).

In [6]:
def paired(a, b):
    va, vb = vectors(a), vectors(b)
    return pd.DataFrame([{**dict(zip(KEYS, k)), **agreement(va[k], vb[k])} for k in va if k in vb])

display(paired(feature["pred"], feature["topk_margin"]).groupby("model")[["spearman", "jaccard"]].mean().reindex(MODELS))
perm = {v: load_importance("permutation", value=v) for v in ("d_RankIC", "topk_change")}
perm = {v: p[(p["pred_len"] == PRED_LEN) & p["universe"].isin(CORE_UNIVERSES)] for v, p in perm.items()}
(paired(perm["d_RankIC"], perm["topk_change"]).groupby("model")[["spearman"]].mean().reindex(MODELS)
 .rename(columns={"spearman": "dRankIC vs top-k change"}))

,spearman,jaccard
model,,
MASTER,1.000,0.965
MATCC,1.000,0.991
FactorVAE,1.000,1.000
HIST,1.000,0.978
DiscoverPLF,1.000,0.978
FinFormer,1.000,1.000


,dRankIC vs top-k change
model,
MASTER,0.258
MATCC,0.175
FactorVAE,0.042
HIST,0.357
DiscoverPLF,0.190
FinFormer,0.012


## 6. Agreement against predictive skill

Per model: the mean test RankIC of its explained runs (notebook 03) next to how stable its explanations are across seeds and how much they agree with the other models. The diagnosis reads this table. Families: across groups only the three shared ones, on which every Alpha360 model agrees perfectly; features: within the group, the finer test.


In [7]:
metrics = collect_results()
metrics = metrics[(metrics["pred_len"] == PRED_LEN) & metrics["universe"].isin(CORE_UNIVERSES)].groupby("model")["RankIC"].mean()
def by_model(p):
    """Mean Spearman per model: with its other seeds, and with the other models (pairs counted for both)."""
    seed_pairs, model_pairs = p[p["factor"] == "seed"], p[p["factor"] == "model"]
    other_models = pd.concat([model_pairs.assign(m=model_pairs["a"]), model_pairs.assign(m=model_pairs["b"])])
    return seed_pairs.groupby("model")["spearman"].mean(), other_models.groupby("m")["spearman"].mean()

(fam_seed, fam_model), (feat_seed, feat_model) = by_model(pairs["family"]), by_model(pairs["feature"])
pd.DataFrame({"mean RankIC": metrics, "families: seeds": fam_seed, "families: other models": fam_model,
              "features: seeds": feat_seed, "features: other models of the group": feat_model}).reindex(MODELS)

,mean RankIC,families: seeds,families: other models,features: seeds,features: other models of the group
MASTER,0.005,0.878,0.886,0.290,0.286
MATCC,0.006,0.948,0.916,0.852,0.287
FactorVAE,-0.009,0.663,0.843,0.268,0.223
HIST,0.012,1.000,0.986,0.956,0.359
DiscoverPLF,0.006,1.000,0.986,0.841,0.343
FinFormer,0.009,1.000,0.986,0.333,0.078


## 7. The horizon

The sections above use $L = 5$. The grid also has $L = 1$ (window $T = 5$) and $L = 20$ ($T = 60$), 252 runs each on the core universes; the horizon fixes both, so two runs that differ only in the horizon also differ in the window. Here the three horizons are pooled: agreement of the runs that differ only in the horizon, next to the other factors; the variance decomposition with the horizon as a fifth factor; and the family shares per model and horizon.

In [8]:
horizon = {level: load_importance(METHOD, "pred", level, FAMILY_VALUE if level == "family" else "importance")
           for level in ("family", "feature")}
horizon = {level: t[t["universe"].isin(CORE_UNIVERSES)] for level, t in horizon.items()}
horizon["family"] = horizon["family"][horizon["family"]["name"] != "market"]
horizon["feature"] = horizon["feature"][[(u, n) not in market for u, n in zip(horizon["feature"]["universe"], horizon["feature"]["name"])]]
print({L: g.groupby(["model", "universe", "seed", "test_year"]).ngroups for L, g in horizon["family"].groupby("pred_len")}, "explained runs per L")
h_pairs = {level: pair_agreement(t) for level, t in horizon.items()}
display(pd.concat({level: p.groupby("factor")[["spearman", "jaccard"]].mean().assign(pairs=p.groupby("factor").size())
                   for level, p in h_pairs.items()}))
by_pair = pd.concat({level: p[p["factor"] == "horizon"].assign(pair=lambda d: d["a"].astype(int).astype(str) + " vs " + d["b"].astype(int).astype(str))
                     .groupby("pair")["spearman"].mean() for level, p in h_pairs.items()}, axis=1)
display(by_pair.rename_axis("L"))
display(pd.concat({level: p[p["factor"] == "horizon"].groupby("model")["spearman"].mean() for level, p in h_pairs.items()}, axis=1).reindex(MODELS))
display(pd.concat({f"{factor}, by L": pd.concat({level: p[p["factor"] == factor].groupby("pred_len")["spearman"].mean()
                                                  for level, p in h_pairs.items()}, axis=1) for factor in ("seed", "model")}))

{1: 252, 5: 252, 20: 252} explained runs per L


spearman  jaccard  pairs
        factor                            
family  horizon      0.838    1.000    756
        model        0.857    1.000   1890
        seed         0.907    1.000    756
        universe     0.866    1.000    702
        year         0.895    1.000   1404
feature horizon      0.594    0.251    756
        model        0.249    0.105    756
        seed         0.585    0.303    756
        universe     0.570    0.256    702
        year         0.641    0.288   1404

,family,feature
L,,
1 vs 20,0.764,0.518
1 vs 5,0.848,0.589
5 vs 20,0.904,0.675


,family,feature
model,,
MASTER,0.715,0.581
MATCC,0.678,0.645
FactorVAE,0.668,0.431
HIST,1.000,0.923
DiscoverPLF,1.000,0.666
FinFormer,0.968,0.319


family  feature
            pred_len                 
seed, by L  1.000      0.893    0.558
            5.000      0.915    0.590
            20.000     0.914    0.607
model, by L 1.000      0.736    0.246
            5.000      0.934    0.263
            20.000     0.903    0.238

In [9]:
for group, title in (("alpha158", "Alpha158 group, its families"), ("alpha360", "Alpha360 group, its families")):
    print(title, "(all horizons)")
    display(decomposition(horizon["family"][horizon["family"]["group"] == group]).set_index("name")
            [["mean_importance", "model", "universe", "year", "horizon", "interactions", "seed", "runs"]])
shares_h = horizon["family"].groupby(["model", "name", "pred_len"])["importance"].mean().unstack("pred_len")
shares_h.reindex(MODELS, level=0)

Alpha158 group, its families (all horizons)


,mean_importance,model,universe,year,horizon,interactions,seed,runs
name,,,,,,,,
candle,0.088,0.078,0.005,0.046,0.468,0.306,0.097,378
momentum,0.178,0.026,0.018,0.090,0.088,0.400,0.378,378
position,0.199,0.040,0.060,0.011,0.006,0.396,0.487,378
price_volume,0.110,0.124,0.146,0.044,0.189,0.407,0.091,378
trend,0.152,0.041,0.010,0.050,0.124,0.261,0.514,378
volatility,0.136,0.079,0.087,0.013,0.116,0.384,0.321,378
volume,0.136,0.058,0.031,0.021,0.073,0.549,0.268,378
all (weighted),NaN,0.058,0.049,0.039,0.123,0.389,0.341,378


Alpha360 group, its families (all horizons)


,mean_importance,model,universe,year,horizon,interactions,seed,runs
name,,,,,,,,
candle,0.033,0.468,0.025,0.017,0.009,0.275,0.205,378
momentum,0.715,0.408,0.020,0.029,0.053,0.244,0.246,378
volume,0.253,0.457,0.030,0.023,0.041,0.231,0.219,378
all (weighted),NaN,0.422,0.023,0.027,0.048,0.241,0.238,378


pred_len                    1     5     20
model       name                          
MASTER      candle       0.133 0.084 0.076
            momentum     0.158 0.182 0.182
            position     0.196 0.209 0.218
            price_volume 0.081 0.095 0.109
            trend        0.169 0.164 0.146
            volatility   0.121 0.127 0.131
            volume       0.141 0.139 0.138
MATCC       candle       0.130 0.080 0.056
            momentum     0.163 0.180 0.185
            position     0.184 0.192 0.203
            price_volume 0.097 0.126 0.138
            trend        0.150 0.151 0.144
            volatility   0.124 0.136 0.148
            volume       0.153 0.135 0.126
FactorVAE   candle       0.084 0.076 0.074
            momentum     0.177 0.197 0.181
            position     0.208 0.194 0.185
            price_volume 0.101 0.115 0.132
            trend        0.174 0.137 0.134
            volatility   0.125 0.150 0.167
            volume       0.132 0.131 0.126
HIST        candle       0.002 0.008 0.010
            momentum     0.646 0.697 0.726
            volume       0.352 0.295 0.264
DiscoverPLF candle       0.067 0.044 0.051
            momentum     0.652 0.685 0.678
            volume       0.281 0.271 0.271
FinFormer   candle       0.039 0.036 0.038
            momentum     0.773 0.785 0.790
            volume       0.188 0.179 0.172

## Diagnosis

All 252 runs of the core grid at L = 5, IG of the score, explained on every rebalance date; point 7 pools the three horizons (756 runs). Families are compared on their net shares, features on their gross importance; in both cases the market inputs are left out. With the day-mean baseline their attribution is zero by construction, and keeping them adds tied zeros that inflate every comparison involving MASTER or MATCC: with them, MASTER's feature-level agreement across seeds was 0.71, without them it is 0.29.

**1. The models agree on the kind of information they use.** Within the Alpha158 group, where all seven families can be compared, two runs that differ in one factor rank the families with a Spearman correlation of 0.74 (another model) to 0.84 (another year), with seeds (0.83) and universes (0.79) in between. Across the two groups only three families are shared, so the comparison is coarse, but it is unanimous: every model, in every universe, puts momentum first, volume second and the candle last. For the Alpha360 models this order largely mirrors the number of columns per family (296, 60 and 4 of 360), so it says less about them than the Alpha158 comparison does. The pooled figures of section 1 (0.90–0.94) are higher only because the Alpha360 models, with these three families alone, always agree perfectly.

**2. They do not agree on the individual features.** Feature-level agreement falls to 0.59 between seeds, 0.60 between universes, 0.66 between years and 0.26 between two models of the same group. Of their top ten features, two models share one on average (Jaccard 0.09). No Alpha158 feature is in the top ten of half of the runs. In Alpha360 only VWAP59 and VWAP9 are, and both are artefacts of the column layout rather than findings about the market: under the reshape of the reference implementations they fall on the last step of the recurrent sequence (PROGETTO.md, Step 7).

**3. For half of the models, a new seed changes the explanation as much as a new architecture.** Section 6 sets each model's feature-level agreement with its own other seeds against its agreement with the other models of its group. The figures are 0.29 against 0.29 for MASTER, 0.27 against 0.22 for FactorVAE, and 0.33 against 0.08 for FinFormer in the other group. For these three models the third case of the diagnosis logic applies: the seeds disagree as much as the architectures, and their explanations cannot carry an architectural conclusion. MATCC, HIST and DiscoverPLF are stable across seeds (0.84–0.96) and still disagree with their group (0.29–0.36), which on its face is the Rashomon pattern. Two observations weaken that reading. First, HIST's stability is not learned: its attributions barely change when its trained weights are replaced by its initial ones (sanity check, notebook 06 and `scripts/evaluate.py`: median Spearman 0.98 over nine runs). Second, the differences within the Alpha360 group follow the order in which the 360 columns reach the recurrent encoder (point 2).

**4. The variance decomposition gives the same picture in other units.** Within Alpha158, the seed accounts for 43% of the variance of the family shares across runs and the interactions for a further 32%. The architecture accounts for 10%, the test year for 9% and the universe for 6%. Within Alpha360 the model accounts for 49% and the seed for 27%. Across all six models the model term rises to 92%, but this term measures the two input sets rather than the architectures. Momentum carries 18–20% of the net share in every Alpha158 model and 69–79% in every Alpha360 model, because 296 of the 360 Alpha360 columns are past prices, which the family mapping assigns to momentum. Relative to that column share (82%), the Alpha360 models actually give momentum less weight than its column count would suggest, and give volume more (25% of the share for 17% of the columns).

**5. Stability does not follow skill.** Every model's mean test RankIC is close to zero: 0.005–0.012, and −0.009 for FactorVAE. FactorVAE has the least stable explanations and no skill, but MASTER (0.005) and FinFormer (0.009) are just as unstable with the same skill as MATCC and DiscoverPLF (0.006), which are stable. Whether a model's explanation survives a change of seed says nothing about whether it ranks stocks better.

**6. The portfolio is explained by the same inputs as the score, but not by the inputs that carry the skill.** IG of the top-k margin ranks the features exactly as IG of the score does (Spearman 1.000, top-10 Jaccard 0.97–1.00). The margin is the score minus a boundary that a stock's own inputs barely move, so their gradients coincide. Permutation importance tells the two apart. Within each run, the ranking of the families by the drop in RankIC when a family is shuffled is at most weakly related to their ranking by the share of the top k it replaces (mean Spearman 0.01 to 0.36 by model), and in 12 of the 30 model–family pairs shuffling even raises the RankIC. The inputs that move the portfolio are not the inputs that make the ranking accurate.

**7. The horizon moves the explanation as much as the architecture (section 7).** Pooling the 756 runs of the three horizons, two runs that differ only in the horizon (and therefore in the window, T = 5, 20 or 60) rank the Alpha158 families at 0.69, as low as two models (0.70) and below two seeds (0.82); their features agree at 0.59, as much as two seeds. Neighbouring horizons agree more: 0.71 between L = 1 and 5, 0.81 between 5 and 20, 0.54 between 1 and 20 on the families. The change is systematic: in MASTER and MATCC the candle falls from 13% of the net share at L = 1 to 6–8% at L = 20, while price-volume and volatility gain one to four points. With the horizon as a fifth factor, it accounts for 12% of the variance of the Alpha158 family shares (47% for the candle), against 34% for the seed, 39% for the interactions and 6% for the model; for the Alpha360 models, whose input window is the same at every horizon, only 5%. Seed agreement does not depend on the horizon (features 0.56, 0.59, 0.61 at L = 1, 5, 20).

**Diagnosis.** The instability comes from the data, and the models only give it its form. Trained on identical inputs, the six architectures extract the same coarse signal: momentum first, then volume, with a predictive skill close to zero. Inside that signal the inputs are near-duplicates of each other: the correlation groups of Alpha158 (chapter 3), and in Alpha360 sixty lags of six series. The data cannot say which duplicate deserves the credit, so the choice is left to whatever else differs between two runs. For MASTER, FactorVAE and FinFormer that is the seed. For the other models it is the architecture, and in Alpha360 the layout of the columns. Notebook 08 adds that consecutive test years of the same seed agree far more than two seeds of the same year (MASTER 0.72 against 0.29, FactorVAE 0.56 against 0.27). Same seed means the same initial weights and the same order of the batches, and that shapes the explanation more than moving the training window by a year. At the level that matters economically, the kind of information, the answer is therefore *data-driven*: every model relies on the same weak signal. At the level of individual features the answer is *undetermined*, because for half of the models the seed is as decisive as the architecture. No architecture can be said to follow a logic of its own.